## Task 3, part 5 - Modelling of a 4th, additional model: protein-similarity kernel regression

This explores a genuinely different algorithm from the Ridge regression used in models 2 and 3: instead of learning a parametric mapping from features to a reduced target space, predict a held-out gene's RNA fingerprint directly as a similarity-weighted average of the 40 training genes' own true fingerprints. Similarity between two perturbations is measured in protein log2FC space (test-time available for a held-out gene, exactly as in model 3) -- genes with more similar protein-level effects are assumed to have more similar RNA-level effects, and contribute proportionally more to the prediction.

Note: this deliberately does *not* reuse Task 2's Leiden perturbation clusters. Those clusters were built by looking at which transcriptional cell-state cluster each perturbation's own cells are enriched in relative to control -- i.e. they are derived from each perturbation's own RNA effect, the very thing we are trying to predict here. Using a held-out gene's own Task 2 cluster label as a model input would be leakage. Protein log2FC avoids this, since it is a different (test-time sanctioned) readout of the same held-out perturbation, not a summary of its RNA effect.

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler

In [2]:
# load back in the RNA fingerprints and split prepared in Task3_01
DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")
train_40 = pd.read_csv(f"{DATA_DIR}/task3_train_40.csv")["perturbation"].tolist()
test_10 = pd.read_csv(f"{DATA_DIR}/task3_test_10.csv")["perturbation"].tolist()

selected_50 = train_40 + test_10
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()

pert_FC_selected.shape

(150, 2042)

## Compute the protein log2FC "fingerprint"

Same computation as in Task3_04: for each condition, compare the mean protein expression of each perturbation's cells to that condition's control cells, on a log2 scale with a pseudocount. 4 of the 24 measured "proteins" are isotype controls (antibody background-binding controls, not real markers) and are dropped, leaving 20 real surface markers.

In [3]:
protein = sc.read_h5ad(f"{DATA_DIR}/protein.h5ad")

# drop isotype controls (antibody background-binding controls, not real markers)
isotype_controls = ["Rat_IgG2a", "Mouse_IgG1", "Mouse_IgG2a", "Mouse_IgG2b"]
protein = protein[:, ~protein.var_names.isin(isotype_controls)].copy()

# keep only control cells and cells belonging to one of the 50 selected perturbations
relevant_mask = protein.obs["perturbation"].isin(selected_50) | (protein.obs["perturbation"] == "control")
protein = protein[relevant_mask.values].copy()

# normalize (no log-transform needed here, log2FC is computed on the normalized scale directly, as in Task3_01)
sc.pp.normalize_total(protein, target_sum=1e4)
protein_norm = np.asarray(protein.X.todense())

protein.shape

/tmp/ipykernel_15175/1216798519.py:12: UserWarning: Some cells have zero counts
  sc.pp.normalize_total(protein, target_sum=1e4)


(94502, 20)

In [4]:
PSEUDOCOUNT = 1.0

# mean normalized protein expression of control cells, per condition
control_means_protein = {}
for cond in conditions:
    mask = (protein.obs["perturbation_2"] == cond) & (protein.obs["perturbation"] == "control")
    control_means_protein[cond] = protein_norm[mask.values].mean(axis=0)

# log2FC protein fingerprint per (perturbation, condition), relative to control cells of that condition
protein_FC = {}
for pert in selected_50:
    for cond in conditions:
        mask = (protein.obs["perturbation_2"] == cond) & (protein.obs["perturbation"] == pert)
        pert_mean = protein_norm[mask.values].mean(axis=0)
        protein_FC[(pert, cond)] = np.log2((pert_mean + PSEUDOCOUNT) / (control_means_protein[cond] + PSEUDOCOUNT))

protein_FC_index = pd.MultiIndex.from_tuples(protein_FC.keys(), names=["perturbation", "condition"])
protein_FC_df = pd.DataFrame(np.vstack(list(protein_FC.values())), index=protein_FC_index, columns=protein.var_names)

protein_FC_df.shape

(150, 20)

## Kernel-weighted similarity prediction

For a query gene in a given condition: measure its Euclidean distance (in standardized protein log2FC space) to each pool gene, convert distances to weights with a Gaussian kernel (closer genes get exponentially more weight), and predict the query's RNA fingerprint as the weighted average of the pool genes' *true* RNA fingerprints -- no PCA, no learned coefficients, just a weighted average. The pool never includes the query gene itself, so this works for both leave-one-out cross-validation and predicting the actual held-out genes.

The kernel bandwidth controls how "local" the average is: a small bandwidth means only the closest few genes matter (like a soft k-NN); a very large bandwidth makes every pool gene's weight roughly equal, so the prediction converges to the same plain training-average as the Task3_02 baseline. This plays the same role that alpha plays for Ridge in models 2 and 3.

In [5]:
def similarity_predict(query_gene, condition, bandwidth, pool_genes):
    """Predict an RNA fingerprint as a kernel-weighted average of pool genes' true RNA fingerprints."""
    # exclude the query gene itself from the pool used to compute the average
    fit_genes = [g for g in pool_genes if g != query_gene]

    # standardize protein features (fit only on the pool) so distance isn't dominated by a few high-variance markers
    scaler = StandardScaler()
    fit_protein = scaler.fit_transform(np.vstack([protein_FC_df.loc[(g, condition)].values for g in fit_genes]))
    query_protein = scaler.transform(protein_FC_df.loc[(query_gene, condition)].values.reshape(1, -1))[0]

    # Euclidean distance from the query gene to each pool gene, in standardized protein space
    distances = np.linalg.norm(fit_protein - query_protein, axis=1)

    # Gaussian kernel: weight decays smoothly with distance, controlled by bandwidth.
    # Computed in log-space and shifted by its own max before exponentiating (a standard numerical-
    # stability trick, the same idea as in softmax): without this, a small bandwidth relative to the
    # actual distances makes every single weight underflow to exactly 0, so they'd all sum to 0 and
    # the normalization below would divide by zero. Subtracting the max first guarantees the closest
    # pool gene always keeps weight exp(0) = 1, so the sum can never be zero, however small the
    # bandwidth is -- in the limit of a tiny bandwidth this just converges to "copy the single nearest
    # neighbor's fingerprint", which is the correct (if extreme) behavior, not a numerical error.
    log_weights = -(distances ** 2) / (2 * bandwidth ** 2)
    weights = np.exp(log_weights - log_weights.max())
    weights /= weights.sum()

    # weighted average of the pool genes' own true RNA fingerprints
    fit_fingerprints = np.vstack([pert_FC_selected.loc[(g, condition)].values for g in fit_genes])
    pred_fingerprint = weights @ fit_fingerprints
    return pred_fingerprint

## Choosing the kernel bandwidth via leave-one-out cross-validation

Hold out one training gene at a time, predict it from the other 39 (per condition), and compare a few candidate bandwidths. This never touches the 10 held-out test genes -- the bandwidth is fixed before we ever look at them.

In [6]:
candidate_bandwidths = [0.1, 0.3, 1, 3, 10, 30, 100, 300]

cv_mse_by_bandwidth = {}
for bandwidth in candidate_bandwidths:
    squared_errors = []
    for cond in conditions:
        for gene in train_40:
            # similarity_predict excludes the query gene itself from the pool, so this is a genuine leave-one-out prediction
            pred = similarity_predict(gene, cond, bandwidth, train_40)
            true = pert_FC_selected.loc[(gene, cond)].values
            squared_errors.append(np.mean((true - pred) ** 2))
    cv_mse_by_bandwidth[bandwidth] = np.mean(squared_errors)

best_bandwidth = min(cv_mse_by_bandwidth, key=cv_mse_by_bandwidth.get)
cv_mse_by_bandwidth, best_bandwidth

({0.1: np.float32(0.003754636),
  0.3: np.float32(0.0036758268),
  1: np.float32(0.002914876),
  3: np.float32(0.0022634785),
  10: np.float32(0.002215495),
  30: np.float32(0.0022207403),
  100: np.float32(0.002221861),
  300: np.float32(0.0022219676)},
 10)

## Predict the held-out test genes and evaluate

Use the chosen bandwidth to predict each of the 10 held-out genes' RNA fingerprint from their own protein log2FC and the 40 training genes' true fingerprints, then evaluate with the same metrics used for the other models so results are directly comparable.

In [7]:
# predict each held-out (gene, condition) pair from the 40 training genes, using the CV-chosen bandwidth
similarity_predictions = {
    (gene, cond): similarity_predict(gene, cond, best_bandwidth, train_40)
    for cond in conditions
    for gene in test_10
}


def evaluate_predictions(true_df, predictions_by_row):
    """Compare each true fingerprint against its predicted fingerprint (looked up per row)."""
    records = []
    for (pert, cond), true_fc in true_df.iterrows():
        pred_fc = predictions_by_row[(pert, cond)]
        pearson_r, _ = pearsonr(true_fc, pred_fc)
        spearman_r, _ = spearmanr(true_fc, pred_fc)
        mse = np.mean((true_fc - pred_fc) ** 2)
        records.append({
            "perturbation": pert,
            "condition": cond,
            "pearson_r": pearson_r,
            "spearman_r": spearman_r,
            "mse": mse,
        })
    return pd.DataFrame(records)


similarity_eval = evaluate_predictions(pert_FC_selected.loc[test_10, :], similarity_predictions)
similarity_eval

,perturbation,condition,pearson_r,spearman_r,mse
0,KCNN4,Control,0.828026,0.454897,0.001511
1,KCNN4,IFNγ,0.845910,0.401296,0.001088
2,KCNN4,Co-culture,0.866200,0.345037,0.001123
3,TIMM50,Control,0.741304,0.391277,0.002770
4,TIMM50,IFNγ,0.625804,0.297666,0.003307
5,TIMM50,Co-culture,0.702722,0.195690,0.003825
6,TXNDC17,Control,0.805096,0.493025,0.004362
7,TXNDC17,IFNγ,0.618780,0.445298,0.004345
8,TXNDC17,Co-culture,0.751722,0.383419,0.004558
9,CORO1A,Control,0.806214,0.375514,0.001195


In [8]:
metrics = ["pearson_r", "spearman_r", "mse"]

# per-condition breakdown (n=10 genes each) -- for biological interpretation
per_condition = similarity_eval.groupby("condition")[metrics].agg(["mean", "std"])

# pooled across all held-out (gene, condition) pairs (n=30) -- single headline number, comparable to the other models
overall = similarity_eval[metrics].agg(["mean", "std"])

per_condition

pearson_r           spearman_r                 mse          
                mean       std       mean       std      mean       std
condition                                                              
Co-culture  0.593845  0.497919   0.267004  0.105432  0.003957  0.005855
Control     0.755218  0.136373   0.399839  0.116983  0.002494  0.001583
IFNγ        0.718614  0.253320   0.378954  0.096558  0.002419  0.001910

In [9]:
overall

,pearson_r,spearman_r,mse
mean,0.689226,0.348599,0.002957
std,0.327971,0.118782,0.003615


## Discussion (initial draft -- please rewrite)

**What this notebook does:** Trains a non-parametric, instance-based model: a held-out gene's RNA fingerprint is predicted as a Gaussian-kernel-weighted average of the 40 training genes' own true fingerprints, where the weights come from how similar the query gene's protein log2FC is to each training gene's protein log2FC (standardized Euclidean distance). Unlike models 2 and 3, there is no PCA target reduction and no learned regression coefficients -- it is literally a weighted average of real, observed fingerprints. The kernel bandwidth (how quickly weight falls off with distance) is chosen via the same leave-one-gene-out cross-validation used throughout.

Note on scope: we deliberately did not build this on Task 2's Leiden perturbation clusters, since those clusters are derived from each perturbation's own observed RNA effect (which cluster of transcriptional cell states its own cells are enriched in) -- using a held-out gene's own cluster label would leak the answer. Protein log2FC is used instead, since it's a genuinely different, test-time-legitimate readout of the same held-out perturbation.

**Results:** LOOCV finds a real interior minimum at bandwidth = 10 (MSE 0.0022155), worse on both sides (0.0037546 at bandwidth=0.1, rising back to 0.0022220 by bandwidth=300 -- the large-bandwidth limit where every training gene gets roughly equal weight, which is mathematically the same plain average the Task3_02 baseline computes). This qualitatively matches model 3's finding (a genuine, if shallow, dip in the CV curve) rather than model 2's (monotonic, no signal at all) -- both models built on protein log2FC find *some* real structure, unlike the ones built on a gene's own baseline expression statistics.

Final test performance: Pearson r = 0.689, Spearman r = 0.349, MSE = 0.00296 -- again essentially indistinguishable from the baseline and both other models. With only 40 training genes to draw a local neighborhood from, even a real similarity signal doesn't reliably beat "predict the average" by more than noise; the best bandwidth found (10) still ends up close to the "average everything" regime, rather than aggressively weighting only a handful of near neighbors.

**Overall picture across all models:** every approach we've tried -- plain averaging, gene-intrinsic expression statistics, protein-based regression, and now protein-based similarity weighting -- converges to almost the same headline numbers (Pearson ~0.69, Spearman ~0.35). The two protein-based models (3 and 4) are the only ones with cross-validation evidence of real signal beyond the shared average, but with 40 training genes that signal is too weak to move the needle much on final test performance. This is a consistent, defensible conclusion for the write-up: exploitable perturbation-specific signal exists but is weak, and more genes (not more model complexity) would likely be the more effective lever for improving on this baseline.